# Parte 2 - API de lluvias

Usamos la API gratuita de Open-Meteo para obtener cuánto llovió en las capitales de los 24 departamentos del Perú y el Callao durante nuestra temporada: del 1 de enero al 31 de mayo de 2025.

In [1]:
import io
import time

import pandas as pd
import requests

## Paso 1: Obtener departamentos y capitales

Wikipedia rechaza `pd.read_html(url)` directamente (error 403). Por eso primero descargamos la página con `requests.get()`, identificándonos como navegador con el encabezado `User-Agent`, y después le pasamos el texto a `pd.read_html`.

In [2]:
url_wikipedia = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

respuesta = requests.get(url_wikipedia, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
print("Estado:", respuesta.status_code)

tablas = pd.read_html(io.StringIO(respuesta.text))
print("Tablas encontradas:", len(tablas))

Estado: 200
Tablas encontradas: 6


In [3]:
# Revisamos el tamaño y las columnas de cada tabla para elegir la correcta
for i in range(len(tablas)):
    print("Tabla", i, "-", len(tablas[i]), "filas -", list(tablas[i].columns))

Tabla 0 - 1 filas - ['Departamentos del Perú (con las provincias de régimen especial)']
Tabla 1 - 24 filas - ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital', 'Ciudad más poblada', 'Superficie (km²)', 'Población (2025)', 'Creación', 'Ubicación']
Tabla 2 - 8 filas - ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
Tabla 3 - 2 filas - ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital', 'Ciudad más poblada', 'Superficie (km²)', 'Población (2025)', 'Creación', 'Ubicación']
Tabla 4 - 6 filas - ['Provincia', 'Capital', 'Creación', 'Desaparición']
Tabla 5 - 1 filas - [0, 1]


La tabla de departamentos es la que tiene las columnas `Departamento`, `Nombre oficial` y `Capital`. Nos quedamos solo con `Departamento` y `Capital`, y las renombramos como `departamento` y `capital`.

In [4]:
for tabla in tablas:
    if "Departamento" in tabla.columns and "Nombre oficial" in tabla.columns:
        tabla_departamentos = tabla

departamentos = tabla_departamentos[["Departamento", "Capital"]].copy()
departamentos.columns = ["departamento", "capital"]
departamentos

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## Paso 2: Verificar las 25 capitales

Debemos tener 24 departamentos + Callao = 25 filas.

In [5]:
print("Filas en la tabla de departamentos:", len(departamentos))

Filas en la tabla de departamentos: 24


**La tabla trae 24 filas, no 25.** Falta el Callao: Wikipedia no lo pone en la tabla de departamentos, sino en otra tabla de la misma página, junto con la Provincia de Lima, como provincia de régimen especial. Agregamos únicamente la fila del Callao, tomándola de esa otra tabla. La Provincia de Lima no se agrega porque la tarea pide solo los 24 departamentos y el Callao.

In [6]:
for tabla in tablas:
    if "Provincia" in tabla.columns and "Nombre oficial" in tabla.columns:
        tabla_regimen_especial = tabla

tabla_regimen_especial[["Provincia", "Nombre oficial", "Capital"]]

,Provincia,Nombre oficial,Capital
0,Callao[nota 6],Provincia Constitucional del Callao,Callao
1,Lima[nota 7],Provincia de Lima,Lima


In [7]:
es_callao = tabla_regimen_especial["Nombre oficial"] == "Provincia Constitucional del Callao"

fila_callao = tabla_regimen_especial[es_callao][["Provincia", "Capital"]].copy()
fila_callao.columns = ["departamento", "capital"]

departamentos = pd.concat([departamentos, fila_callao], ignore_index=True)
print("Filas después de agregar el Callao:", len(departamentos))

Filas después de agregar el Callao: 25


Ahora revisamos los nombres. Algunas celdas traen texto extra que impediría buscar la ciudad en Open-Meteo: notas al pie entre corchetes, como `Callao[nota 6]`, y aclaraciones entre paréntesis, como `Huacho (de facto)`. Quitamos ese texto y mostramos qué se corrigió.

In [8]:
for i in range(len(departamentos)):
    departamento = departamentos.loc[i, "departamento"]
    capital = departamentos.loc[i, "capital"]

    # Quitamos lo que está desde "[" (notas al pie) o desde "(" (aclaraciones)
    departamento_limpio = departamento.split("[")[0]
    departamento_limpio = departamento_limpio.split("(")[0].strip()

    capital_limpia = capital.split("[")[0]
    capital_limpia = capital_limpia.split("(")[0].strip()

    if departamento_limpio != departamento:
        print("Departamento corregido:", departamento, "->", departamento_limpio)
    if capital_limpia != capital:
        print("Capital corregida:", capital, "->", capital_limpia)

    departamentos.loc[i, "departamento"] = departamento_limpio
    departamentos.loc[i, "capital"] = capital_limpia

Capital corregida: Huacho (de facto) -> Huacho
Departamento corregido: Callao[nota 6] -> Callao


**Decisión para Lima:** Wikipedia indica *Huacho (de facto)* como capital del departamento de Lima, porque en Huacho funciona la sede del Gobierno Regional de Lima. La ciudad de Lima aparece en la otra tabla como capital de la Provincia de Lima, que no es uno de los 25 registros. Por eso usamos **Huacho** para el departamento de Lima.

In [9]:
print("Total de filas:", len(departamentos))
print("Departamentos distintos:", departamentos["departamento"].nunique())

if len(departamentos) == 25:
    print("Correcto: 24 departamentos + Callao = 25 filas")
else:
    print("Revisar: la tabla no tiene 25 filas")

departamentos

Total de filas: 25
Departamentos distintos: 25
Correcto: 24 departamentos + Callao = 25 filas


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## Paso 3: Obtener coordenadas

Usamos la API de geocodificación de Open-Meteo. Una búsqueda puede devolver varios lugares con el mismo nombre. Por ejemplo, esto devuelve la API para "Arequipa":

In [10]:
url_geocodificacion = "https://geocoding-api.open-meteo.com/v1/search"

parametros = {"name": "Arequipa", "count": 10, "language": "es"}
respuesta = requests.get(url_geocodificacion, params=parametros, timeout=30)

ejemplo = pd.DataFrame(respuesta.json()["results"])
ejemplo[["name", "country_code", "admin1", "latitude", "longitude"]]

,name,country_code,admin1,latitude,longitude
0,Arequipa,PE,Departamento de Arequipa,-16.39899,-71.53747
1,Arequipa,PE,Departamento de Cajamarca,-5.51057,-78.61048
2,Arequipa,BO,Departamento de La Paz,-15.60000,-67.95000
3,Arequipa,BO,Departamento del Beni,-12.36667,-66.28333
4,Arequipa,PE,Departamento de Loreto,-4.62417,-73.89528
5,Arequipa,PE,Departamento de San Martín,-6.64770,-76.72495
6,Arequipa,PE,Departamento de Ayacucho,-12.43336,-73.98249
7,Arequipa Alta,PE,Departamento de Ayacucho,-12.63109,-73.87136
8,Arequipa Baja,PE,Departamento de Ayacucho,-12.62230,-73.87136


Hay lugares llamados Arequipa en Bolivia y también en otros departamentos del Perú (Cajamarca, Loreto, San Martín, Ayacucho). Por eso no tomamos automáticamente el primer resultado: recorremos los resultados y elegimos el primero que cumpla dos condiciones:

1. `country_code` es `"PE"`.
2. `admin1` corresponde al departamento.

`admin1` no siempre está escrito igual que en Wikipedia: puede venir como `Departamento de Cusco`, como `Ancash` (sin tilde) o como `Provincia Constitucional del Callao`. Para comparar, normalizamos los dos nombres: pasamos a minúsculas, quitamos "departamento de", "departamento del" y "provincia constitucional del", y quitamos las tildes.

In [11]:
def normalizar(nombre):
    nombre = nombre.lower()
    nombre = nombre.replace("departamento del ", "")
    nombre = nombre.replace("departamento de ", "")
    nombre = nombre.replace("provincia constitucional del ", "")
    nombre = nombre.replace("á", "a")
    nombre = nombre.replace("é", "e")
    nombre = nombre.replace("í", "i")
    nombre = nombre.replace("ó", "o")
    nombre = nombre.replace("ú", "u")
    return nombre.strip()


# Probamos con los casos que vimos en la API
print(normalizar("Departamento de Cusco"), "|", normalizar("Cusco"))
print(normalizar("Ancash"), "|", normalizar("Áncash"))
print(normalizar("Provincia Constitucional del Callao"), "|", normalizar("Callao"))

cusco | cusco
ancash | ancash
callao | callao


In [12]:
resultados_geo = []
sin_resultado = []

for i in range(len(departamentos)):
    departamento = departamentos.loc[i, "departamento"]
    capital = departamentos.loc[i, "capital"]

    parametros = {"name": capital, "count": 10, "language": "es"}
    respuesta = requests.get(url_geocodificacion, params=parametros, timeout=30)
    respuesta.raise_for_status()
    lugares = respuesta.json().get("results", [])

    # Buscamos el primer lugar que sea del Perú y del departamento correcto
    elegido = None
    for lugar in lugares:
        es_peru = lugar.get("country_code") == "PE"
        mismo_departamento = normalizar(lugar.get("admin1", "")) == normalizar(departamento)
        if es_peru and mismo_departamento:
            elegido = lugar
            break

    if elegido is None:
        print(capital, "- NO se encontró un resultado del Perú en", departamento)
        sin_resultado.append(capital)
        resultados_geo.append({
            "departamento": departamento,
            "capital": capital,
            "admin1": None,
            "latitud": None,
            "longitud": None,
        })
    else:
        print(capital, "- resultados de la API:", len(lugares), "- elegido:", elegido["name"], "|", elegido["admin1"])
        resultados_geo.append({
            "departamento": departamento,
            "capital": capital,
            "admin1": elegido["admin1"],
            "latitud": elegido["latitude"],
            "longitud": elegido["longitude"],
        })

    time.sleep(1)

print()
print("Capitales sin resultado correcto:", len(sin_resultado), sin_resultado)

Chachapoyas - resultados de la API: 4 - elegido: Chachapoyas | Departamento de Amazonas


Huaraz - resultados de la API: 1 - elegido: Huaraz | Ancash


Abancay - resultados de la API: 1 - elegido: Abancay | Apurímac


Arequipa - resultados de la API: 9 - elegido: Arequipa | Departamento de Arequipa


Ayacucho - resultados de la API: 10 - elegido: Ayacucho | Departamento de Ayacucho


...

Tacna - resultados de la API: 10 - elegido: Tacna | Departamento de Tacna


Tumbes - resultados de la API: 6 - elegido: Tumbes | Departamento de Tumbes


Pucallpa - resultados de la API: 7 - elegido: Pucallpa | Ucayali


Callao - resultados de la API: 10 - elegido: El Callao | Provincia Constitucional del Callao



Capitales sin resultado correcto: 0 []


## Paso 4: Verificar las coordenadas

Tabla de verificación: para cada departamento se ve la capital buscada y el `admin1` que devolvió la API, junto con las coordenadas elegidas.

In [13]:
coordenadas = pd.DataFrame(resultados_geo)
coordenadas = coordenadas[["departamento", "capital", "admin1", "latitud", "longitud"]]
coordenadas

,departamento,capital,admin1,latitud,longitud
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422


In [14]:
print("Filas en la tabla de verificación:", len(coordenadas))

# Volvemos a comparar admin1 con el departamento, fila por fila
coinciden = 0
for i in range(len(coordenadas)):
    admin1 = coordenadas.loc[i, "admin1"]
    departamento = coordenadas.loc[i, "departamento"]
    if admin1 is not None and normalizar(admin1) == normalizar(departamento):
        coinciden = coinciden + 1
    else:
        print("No coincide:", departamento, "|", admin1)

print("Filas donde admin1 corresponde al departamento:", coinciden, "de", len(coordenadas))
print("Latitudes vacías:", coordenadas["latitud"].isna().sum())
print("Longitudes vacías:", coordenadas["longitud"].isna().sum())

Filas en la tabla de verificación: 25
Filas donde admin1 corresponde al departamento: 25 de 25
Latitudes vacías: 0
Longitudes vacías: 0


**Verificación de la geocodificación:**

- La API puede devolver varias ciudades con el mismo nombre. Por ejemplo, "Arequipa" devolvió 9 lugares: dos en Bolivia y seis en otros departamentos del Perú. Varias capitales, como Ayacucho, Trujillo o Callao, devolvieron 10 resultados.
- Solo se aceptaron resultados con `country_code` igual a `"PE"`.
- Se verificó que `admin1` correspondiera al departamento: las 25 filas coinciden y ninguna latitud o longitud quedó vacía.
- Se consideraron las diferencias de escritura: `admin1` aparece a veces con prefijo (`Departamento de Cusco`, `Provincia Constitucional del Callao`), otras sin él (`Apurímac`, `Puno`, `Ucayali`) y otras sin tilde (`Ancash`). La normalización del paso 3 resolvió todos estos casos.
- La API escribe dos capitales de otra forma: Cusco aparece como "Cuzco" y Callao como "El Callao". Son la misma ciudad, en el departamento correcto.

## Paso 5: Obtener datos de lluvia

Para cada capital pedimos a la API histórica de Open-Meteo la lluvia diaria (`precipitation_sum`) entre el 1 de enero y el 31 de mayo de 2025, en hora de Lima. Entre cada pedido esperamos 1 segundo.

In [15]:
url_historico = "https://archive-api.open-meteo.com/v1/archive"

lluvias_diarias = {}   # para cada departamento, la lista de lluvia de cada día

for i in range(len(coordenadas)):
    departamento = coordenadas.loc[i, "departamento"]

    parametros = {
        "latitude": coordenadas.loc[i, "latitud"],
        "longitude": coordenadas.loc[i, "longitud"],
        "start_date": "2025-01-01",
        "end_date": "2025-05-31",
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    respuesta = requests.get(url_historico, params=parametros, timeout=60)
    respuesta.raise_for_status()
    datos = respuesta.json()

    fechas = datos["daily"]["time"]
    precipitaciones = datos["daily"]["precipitation_sum"]
    lluvias_diarias[departamento] = precipitaciones

    print(departamento, "- días recibidos:", len(precipitaciones), "- del", fechas[0], "al", fechas[-1])

    time.sleep(1)

print()
print("Consultas realizadas:", len(lluvias_diarias))

Amazonas - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Áncash - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Apurímac - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Arequipa - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Ayacucho - días recibidos: 151 - del 2025-01-01 al 2025-05-31


...

Tacna - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Tumbes - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Ucayali - días recibidos: 151 - del 2025-01-01 al 2025-05-31


Callao - días recibidos: 151 - del 2025-01-01 al 2025-05-31



Consultas realizadas: 25


Cada consulta trae 151 días: 31 de enero + 28 de febrero + 31 de marzo + 30 de abril + 31 de mayo.

## Paso 6: Calcular indicadores

Para cada departamento calculamos:

- `lluvia_total_mm`: la suma de la lluvia de todos los días de la temporada.
- `dias_lluvia_fuerte`: la cantidad de días con 20 mm o más.

Si algún día llega sin dato (`None`), no se suma ni se cuenta. No lo convertimos a 0, porque no sabemos si ese día llovió o no. En el paso 7 revisamos cuántos hubo.

In [16]:
indicadores = []

for departamento in lluvias_diarias:
    precipitaciones = lluvias_diarias[departamento]

    lluvia_total = 0
    dias_fuertes = 0

    for valor in precipitaciones:
        if valor is not None:
            lluvia_total = lluvia_total + valor
            if valor >= 20:
                dias_fuertes = dias_fuertes + 1

    indicadores.append({
        "departamento": departamento,
        "lluvia_total_mm": round(lluvia_total, 1),
        "dias_lluvia_fuerte": dias_fuertes,
    })

tabla_indicadores = pd.DataFrame(indicadores)
tabla_indicadores

,departamento,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,427.9,0
1,Áncash,1415.9,11
2,Apurímac,820.0,3
3,Arequipa,439.4,3
4,Ayacucho,500.8,1
5,Cajamarca,875.7,2
6,Cusco,801.3,4
7,Huancavelica,930.4,4
8,Huánuco,617.0,4
9,Ica,124.3,2


## Paso 7: Verificar datos faltantes

Contamos cuántos valores de `precipitation_sum` llegaron como `None` en las 25 consultas.

In [17]:
dias_revisados = 0
dias_sin_dato = 0

for departamento in lluvias_diarias:
    nulos_departamento = 0
    for valor in lluvias_diarias[departamento]:
        dias_revisados = dias_revisados + 1
        if valor is None:
            nulos_departamento = nulos_departamento + 1

    if nulos_departamento > 0:
        print(departamento, "- días sin dato:", nulos_departamento)

    dias_sin_dato = dias_sin_dato + nulos_departamento

print("Consultas revisadas:", len(lluvias_diarias))
print("Días revisados:", dias_revisados)
print("Días sin dato:", dias_sin_dato)

Consultas revisadas: 25
Días revisados: 3775
Días sin dato: 0


Hicimos la comprobación en los 3775 valores recibidos (25 capitales × 151 días) y **no se encontraron días sin datos**: ningún valor llegó como `None`. Por eso `lluvia_total_mm` y `dias_lluvia_fuerte` se calcularon con los 151 días de cada capital. Si hubieran aparecido `None`, el cálculo del paso 6 los habría dejado fuera de la suma y del conteo, sin convertirlos a 0.

## Paso 8: Guardar resultados

Unimos las coordenadas con los indicadores y dejamos las columnas en el orden pedido.

In [18]:
tabla_final = coordenadas.merge(tabla_indicadores, on="departamento")
tabla_final = tabla_final[["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]]

In [19]:
departamentos_esperados = [
    "Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
    "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
    "Ica", "Junín", "La Libertad", "Lambayeque", "Lima",
    "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura",
    "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali",
]

faltantes = []
for nombre in departamentos_esperados:
    if nombre not in list(tabla_final["departamento"]):
        faltantes.append(nombre)

print("Filas:", len(tabla_final))
print("Columnas:", list(tabla_final.columns))
print("Departamentos faltantes (24 departamentos + Callao):", faltantes)
print("Coordenadas vacías:", tabla_final["latitud"].isna().sum() + tabla_final["longitud"].isna().sum())
print("Valores vacíos en lluvia_total_mm:", tabla_final["lluvia_total_mm"].isna().sum())
print("Valores vacíos en dias_lluvia_fuerte:", tabla_final["dias_lluvia_fuerte"].isna().sum())

Filas: 25
Columnas: ['departamento', 'capital', 'latitud', 'longitud', 'lluvia_total_mm', 'dias_lluvia_fuerte']
Departamentos faltantes (24 departamentos + Callao): []
Coordenadas vacías: 0
Valores vacíos en lluvia_total_mm: 0
Valores vacíos en dias_lluvia_fuerte: 0


In [20]:
tabla_final

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0
1,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11
2,Apurímac,Abancay,-13.63426,-72.88380,820.0,3
3,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3
4,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1
5,Cajamarca,Cajamarca,-7.16378,-78.50027,875.7,2
6,Cusco,Cusco,-13.53188,-71.96701,801.3,4
7,Huancavelica,Huancavelica,-12.78691,-74.97298,930.4,4
8,Huánuco,Huánuco,-9.92882,-76.23989,617.0,4
9,Ica,Ica,-14.07538,-75.73422,124.3,2


In [21]:
tabla_final.to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")

# Leemos el archivo guardado para comprobarlo
revision = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")
print("Filas en el CSV:", len(revision))
print("Columnas en el CSV:", list(revision.columns))

Filas en el CSV: 25
Columnas en el CSV: ['departamento', 'capital', 'latitud', 'longitud', 'lluvia_total_mm', 'dias_lluvia_fuerte']
